## Imports

In [ ]:
from pathlib import Path
from loguru import logger
from scapy.all import IP, TCP, Ether, raw
from scapy.utils import RawPcapReader
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# Set global font family and size
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif", "serif"]
mpl.rcParams["font.size"] = 12  # Adjust per Springer guidelines
mpl.rcParams["axes.labelsize"] = 12
mpl.rcParams["axes.titlesize"] = 13
mpl.rcParams["xtick.labelsize"] = 11
mpl.rcParams["ytick.labelsize"] = 11
mpl.rcParams["legend.fontsize"] = 11

# Optional: PDF/LaTeX text rendering for enhanced output
mpl.rcParams["text.usetex"] = False

In [ ]:
def stream_pcap(pcap_path):
    """Generator to yield parsed packets and their timestamps from a pcap file"""
    for pkt_data, pkt_metadata in RawPcapReader(pcap_path):
        pkt = Ether(pkt_data)
        ts = pkt_metadata.sec + pkt_metadata.usec / 1e6
        yield pkt, ts


## Total Packets in CSV file vs Total Packets matched

In [ ]:
import numpy as np

# bar plot with flow label in x and average total matched and total labelled in y groupped plot
# 'total_matched_pkts', 'total_labeled_pkts'
# for any bars, write its value on top of it


def plot_matched_labelled_packets(df, data_name, save_path=None):
    avg_df = (
        df.groupby("flow_label")[["total_matched_pkts", "total_labeled_pkts"]]
        .mean()
        .reset_index()
    )
    avg_df.flow_label = avg_df.flow_label.apply(lambda x: x.replace("_", "\n"))
    match_rate = avg_df["total_matched_pkts"].sum() / avg_df["total_labeled_pkts"].sum()
    # plot it

    p = avg_df.melt(
        id_vars=["flow_label"],
        value_vars=["total_matched_pkts", "total_labeled_pkts"],
        var_name="packet_type",
        value_name="packet_count",
    )
    plt.figure(figsize=(15, 6))
    sns.barplot(data=p, x="flow_label", y="packet_count", hue="packet_type")
    for index, row in avg_df.iterrows():
        plt.text(
            index - 0.2,
            row["total_matched_pkts"] + 1,
            round(row["total_matched_pkts"], 2),
            color="black",
            ha="center",
        )
        plt.text(
            index + 0.2,
            row["total_labeled_pkts"] + 1,
            round(row["total_labeled_pkts"], 2),
            color="black",
            ha="center",
        )
    plt.title(
        f"Average Matched/Labelled Packet Counts per Flow Label - {data_name.upper()} | Match Rate: {match_rate:.2%}"
    )
    plt.xlabel("Flow Label")
    plt.ylabel("Average Packet Count")
    if save_path:
        plt.savefig(
            Path(save_path)
            / f"avg_matched_labelled_packets_per_flow_label_{data_name}.pdf",
            dpi=300,
            bbox_inches="tight",
        )
    plt.show()


# plot avg raw_bytes_avg_length per flow label
def plot_avg_raw_bytes_length(df, data_name, save_path=None):
    avg_df = df.groupby("flow_label")[["raw_bytes_avg_length"]].mean().reset_index()
    avg_df.flow_label = avg_df.flow_label.apply(lambda x: x.replace("_", "\n"))
    # plot it
    plt.figure(figsize=(15, 6))
    sns.barplot(data=avg_df, x="flow_label", y="raw_bytes_avg_length")
    for index, row in avg_df.iterrows():
        plt.text(
            index,
            row["raw_bytes_avg_length"] + 2,
            round(row["raw_bytes_avg_length"], 2),
            color="black",
            ha="center",
        )
    plt.title(f"Average Raw Bytes Length per Flow Label - {data_name.upper()}")
    plt.xlabel("Flow Label")
    plt.ylabel("Average Raw Bytes Length")
    if save_path:
        plt.savefig(
            Path(save_path) / f"avg_raw_bytes_length_per_flow_label_{data_name}.pdf",
            dpi=300,
            bbox_inches="tight",
        )
    plt.show()


def plot_match_rates(df, data_name, save_path=None):
    avg_df = df.groupby("flow_label")[["match_rate"]].mean().reset_index()
    avg_df.flow_label = avg_df.flow_label.apply(lambda x: x.replace("_", "\n"))
    # plot it
    plt.figure(figsize=(15, 6))
    sns.barplot(data=avg_df, x="flow_label", y="match_rate")
    for index, row in avg_df.iterrows():
        plt.text(
            index,
            row["match_rate"] + 0.01,
            f"{row['match_rate']:.2f}%",
            color="black",
            ha="center",
        )
    plt.title(f"Average Match Rate per Flow Label - {data_name.upper()}")
    plt.xlabel("Flow Label")
    plt.ylabel("Average Match Rate")
    if save_path:
        plt.savefig(
            Path(save_path) / f"avg_match_rate_per_flow_label_{data_name}.pdf",
            dpi=300,
            bbox_inches="tight",
        )
    plt.show()


def plot_forward_backward_match_rates(df, data_name, save_path=None):
    avg_df = (
        df.groupby("flow_label")[["forward_match_rate", "backward_match_rate"]]
        .mean()
        .reset_index()
    )
    avg_df.flow_label = avg_df.flow_label.apply(lambda x: x.replace("_", "\n"))
    # plot it
    p = avg_df.melt(
        id_vars=["flow_label"],
        value_vars=["forward_match_rate", "backward_match_rate"],
        var_name="direction",
        value_name="match_rate",
    )
    plt.figure(figsize=(15, 6))
    sns.barplot(data=p, x="flow_label", y="match_rate", hue="direction")
    for index, row in avg_df.iterrows():
        plt.text(
            index - 0.2,
            row["forward_match_rate"] + 0.01,
            f"{row['forward_match_rate']:.2f}%",
            color="black",
            ha="center",
        )
        plt.text(
            index + 0.2,
            row["backward_match_rate"] + 0.01,
            f"{row['backward_match_rate']:.2f}%",
            color="black",
            ha="center",
        )
    plt.title(
        f"Average Forward/Backward Match Rates per Flow Label - {data_name.upper()}"
    )
    plt.xlabel("Flow Label")
    plt.ylabel("Average Match Rate")
    if save_path:
        plt.savefig(
            Path(save_path)
            / f"avg_forward_backward_match_rates_per_flow_label_{data_name}.pdf",
            dpi=300,
            bbox_inches="tight",
        )
    plt.show()


def plot_labels_per_session(df, data_name, save_path=None):
    label_counts = df["flow_label"].value_counts().reset_index()
    label_counts.columns = ["flow_label", "count"]
    label_counts.flow_label = label_counts.flow_label.apply(
        lambda x: x.replace("_", "\n")
    )

    plt.figure(figsize=(15, 6))
    sns.barplot(data=label_counts, x="flow_label", y="count")
    for index, row in label_counts.iterrows():
        plt.text(index, row["count"] + 2, row["count"], color="black", ha="center")
    plt.title(f"Number of Sessions per Flow Label - {data_name.upper()}")
    plt.xlabel("Flow Label")
    plt.ylabel("Number of Sessions")
    if save_path:
        plt.savefig(
            Path(save_path) / f"num_sessions_per_flow_label_{data_name}.pdf",
            dpi=300,
            bbox_inches="tight",
        )
    plt.show()


save_path = r"rosaid\assets\figures"

data_paths = [
    # r"rosaid\data\iec104_sessions\labelled_sessions.csv",
    # r"rosaid\data\120_timeout_dnp3_sessions\labelled_sessions.csv",
    r"rosaid\data\rosids23_sessions\labelled_sessions.csv",
]
for data_path in data_paths:
    data_name = (
        "dnp3"
        if "dnp3" in data_path
        else "iec104"
        if "iec104" in data_path
        else "rosids23"
    )
    df = pd.read_csv(data_path)
    df.columns = df.columns.str.strip()
    df["match_rate"] = 100 * df["total_matched_pkts"] / df["total_labeled_pkts"]
    df["backward_match_rate"] = (
        100 * df["matched_backward_pkts"] / df["labled_backward_pkts"]
    )
    df["forward_match_rate"] = (
        100 * df["matched_forward_pkts"] / df["labled_forward_pkts"]
    )
    df = df.query("match_rate>0").copy()
    plot_matched_labelled_packets(df, data_name, save_path=save_path)
    plot_avg_raw_bytes_length(df, data_name, save_path=save_path)
    plot_match_rates(df, data_name, save_path=save_path)
    plot_labels_per_session(df, data_name, save_path=save_path)
    # plot_forward_backward_match_rates(df, data_name, save_path=save_path)


## IEC104

In [ ]:
data_root = Path(
    r"rosaid\data\iec104\attack-data"
)
pcap_file = "capture104-floodattack.pcap"

pcap_path = data_root / pcap_file
csv_path = pcap_path.parent / pcap_file.replace("pcap", "csv")

all_packets = [p for p in stream_pcap(str(pcap_path))]

**capture104-floodattack.pcap**: Focus on the packet: `Frame 11: 60 bytes on wire (480 bits), 60 bytes captured (480 bits)` and at its last: `IEC 60870-5-104: <- U (STARTDT act)`.

Hexdump:
```
0000   a6 1c 4e ca 31 fc 8a a7 27 61 82 b1 08 00 45 00
0010   00 2e 1b d9 40 00 40 06 f7 c5 79 8e 1a 05 79 8e
0020   1a 0a 09 64 b0 2d 93 ff 51 82 30 6f 73 24 50 18
0030   01 f6 27 4c 00 00 68 04 0b 00 00 00
```

It seems the iec frame is passed as a payload.

In [ ]:
pkt = all_packets[10][0]
pkt.show()

In [ ]:

## sourc/dest ip/port
arr = np.frombuffer(raw(pkt), dtype=np.uint8)

In [ ]:
raw(pkt)


### Packets related to the session

In [ ]:
df = pd.read_csv(csv_path)
df["timestamp"] = pd.to_datetime(df["Timestamp"], format="%d/%m/%Y %I:%M:%S %p")
df = df.sort_values(by="timestamp", ascending=True)

df.head()


In [ ]:
(
    pd.to_datetime(float(all_packets[0][1]) * 10**9),
    pd.to_datetime(float(all_packets[-1][1]) * 10**9),
    df["timestamp"].min(),
    df["timestamp"].max(),
)

In [ ]:
# pkt_time1 = pd.to_datetime(float(all_packets[0].time) * 10**9)
# pkt_time2 = pd.to_datetime(float(all_packets[-1].time) * 10**9)
pkt_time1 = pd.to_datetime(float(all_packets[0][1]) * 10**9)
pkt_time2 = pd.to_datetime(float(all_packets[-1][1]) * 10**9)
df_time1 = df["timestamp"].min()
df_time2 = df["timestamp"].max()

time_diff1 = df_time1 - pkt_time1
time_diff2 = df_time2 - pkt_time2
avg_diff = (time_diff1 + time_diff2) / 2

print(
    f"PKT time range: {pkt_time1} to {pkt_time2}\n"
    f"DFm time range: {df_time1} to {df_time2}"
)
print(f"Time difference for first packet: {time_diff1.total_seconds()} seconds")
print(f"Time difference for last packet: {time_diff2.total_seconds()} seconds")
print(f"Average time difference: {avg_diff}")


In [ ]:
# only if labelled csv available
ldf = pd.read_csv(
    r"rosaid\data\iec104_sessions\labelled_sessions.csv"
)
ldf["match_rate"] = ldf["total_matched_pkts"] / ldf["total_labeled_pkts"]
focus_df = (
    ldf.query("flow_label=='floodattack'")
    # .sort_values(by="match_rate")
    .copy()
    .reset_index(drop=True)
)

# take first row
focus_row = focus_df.iloc[0]
start_time = pd.to_datetime(focus_row["start_time"]).timestamp()
end_time = pd.to_datetime(focus_row["end_time"]).timestamp()
focus_row

In [ ]:
ldf.query('flow_label=="floodattack"').head(8)[
    ["start_time", "end_time", "total_labeled_pkts", "total_matched_pkts", "match_rate"]
]

In [ ]:
focus_row = ldf.query("flow_label=='floodattack'").reset_index().iloc[0]
start_time = pd.to_datetime(focus_row["start_time"])
start_ts = start_time.timestamp()
end_time = pd.to_datetime(focus_row["end_time"])
end_ts = end_time.timestamp()
real_duration_microsec = 2041135.0
end_real = start_time + pd.Timedelta(microseconds=real_duration_microsec + 1)
end_ts_real = end_real.timestamp()
print(
    f"Start time: {start_time}, End time: {end_time}|{end_real}, Pkt: {all_packets[0][1]}"
)
pkts = [pkt[0] for pkt in all_packets if start_ts <= float(pkt[1]) <= end_ts_real]
len(pkts), pkts

In [ ]:
pkts_data = []
i = 0
for pkt_data, pkt_metadata in RawPcapReader(str(pcap_path)):
    epoch = pkt_metadata.sec + pkt_metadata.usec / 1e6
    ts = pd.to_datetime(epoch, unit="s")
    pkt = Ether(pkt_data)

    pkts_data.append((pkt, pkt_metadata, ts, epoch))
    print(epoch, pd.to_datetime(pkt.time, unit="s"), ts, pkt.summary())
    i += 1
    if i == 5:
        break

In [ ]:
from scapy.all import rdpcap
all_pkts = rdpcap(str(pcap_path))

In [ ]:
pkts_data = []
i = 0
for pkt in all_pkts:
    ts = pd.to_datetime(float(pkt.time) * 10**6, unit="us")
    pkts_data.append([ts, pkt])
    print(ts, pkt.summary())
    i += 1
    if i == 5:
        break


In [ ]:
end_time = pd.to_datetime(
    pkts_data[0][0].timestamp() * 10**6 + real_duration_microsec, unit="us"
)
end_ts = end_time.timestamp()
pkts_data[0][0].timestamp(), end_ts

In [ ]:
pkts = [pkt[0] for pkt in all_packets if start_ts <= float(pkt[1]) <= end_ts]
len(pkts), pkts


In [ ]:
# this ts has fractional microseconds precision so only use the date and time till secs
# find fractional microseconds difference
fractional_microsec_diff = []

for pkt, ts in all_packets:
    dts = pd.to_datetime(ts, unit="s")
    dts_trunc = dts.replace(microsecond=0)
    diff_microsec = (dts - dts_trunc).microseconds
    fractional_microsec_diff.append(diff_microsec)
# ts = all_packets[0][1]
# dts = pd.to_datetime(ts, unit="s")
# dts.replace(microsecond=0).timestamp(), ts


In [ ]:
import numpy as np

fractional_microsec_diff = np.array(fractional_microsec_diff)

In [ ]:
fractional_microsec_diff.mean()

### Test DoS Attack

In [ ]:
df = pd.read_csv(
    r"rosaid\data\iec104\attack-data\capture104-dosattack.csv"
)
df.shape

In [ ]:
df.columns

In [ ]:
df["Flow Duration"].describe()

In [ ]:
df["total_pkts"] = df["Total Bwd packets"] + df["Total Fwd Packet"]
# histogram
df.total_pkts.unique()  # .describe()

In [ ]:
df.total_pkts.value_counts().sort_index()

In [ ]:
ridx = 0
for idx, row in df.query("total_pkts>=5").iterrows():
    total_pkts = row["total_pkts"]
    print(idx)
    if ridx > 5:
        break
    ridx += 1

In [ ]:
# rosaid\data\iec104\attack-data\capture104-mitmattack.pcap

data_root = Path(
    r"rosaid\data\iec104\attack-data"
)
pcap_file = "capture104-dosattack.pcap"

pcap_path = data_root / pcap_file

all_packets = [p for p in stream_pcap(str(pcap_path))]

In [ ]:
len(all_packets)

In [ ]:
ndf = pd.read_csv(
    r"rosaid\data\20200514_Disable_Unsolicited_Messages_Attack_UOWM_DNP3_Dataset_Attacker_01.pcapDNP3_FLOWLABELED.csv"
)
ndf.columns = ndf.columns.str.strip()
ndf.columns

In [ ]:
# convert to numeric

pkt = all_packets[0][0]
bytes_pkt = raw(pkt)
list_bytes = list(bytes_pkt)

max(list_bytes)

In [ ]:
df.flow_label.str.upper().unique()

In [ ]:
total_lbld = df.total_labeled_pkts.values

# its pctl in 10 diff
for p in range(0, 101, 3):
    val = np.percentile(total_lbld, p)
    print(f"{p}th percentile: {val}")

Now view real distribution as well.

In [ ]:
csv_root = Path(r"rosaid\data\iec104")
csv_files = list(csv_root.rglob("*.csv"))

adf = pd.DataFrame()
for csv_file in csv_files:
    temp_df = pd.read_csv(csv_file)
    temp_df["source_file"] = str(csv_file.name)
    adf = pd.concat([adf, temp_df], ignore_index=True)
adf.to_csv(csv_root.parent / "iec104_combined.csv", index=False)


In [ ]:
adf

### View Total Packets

In [ ]:
df = pd.read_csv(
    r"rosaid\data\iec104_combined.csv"
)
df["total_pkts"] = df["Total Fwd Packet"] + df["Total Bwd packets"]

In [ ]:
import numpy as np

pctl_values = [80, 90, 93, 95, 99]
# for all labels find pctl values
pctl_df = pd.DataFrame(
    columns=[
        "label",
        "num_sessions",
        *[f"pctl_{pctl}" for pctl in pctl_values],
        "max_pkts",
        *[f"nsess_pctl_{pctl}" for pctl in pctl_values],
    ]
)

for key, group in df.groupby("Label"):
    pctl_row = [key, len(group)]
    nsess = []
    for pctl in pctl_values:
        value = np.percentile(group["total_pkts"], pctl)
        pctl_row.append(value)
        nsess.append(len(group[group["total_pkts"] <= value]))

    pctl_row.append(group["total_pkts"].max())
    pctl_row.extend(nsess)
    pctl_df.loc[len(pctl_df)] = pctl_row

# add for all labels combined
pctl_row = ["All Labels", len(df)]
nsess = []
for pctl in pctl_values:
    value = np.percentile(df["total_pkts"], pctl)
    pctl_row.append(value)
    nsess.append(len(df[df["total_pkts"] <= value]))
pctl_row.append(df["total_pkts"].max())
pctl_row.extend(nsess)
pctl_df.loc[len(pctl_df)] = pctl_row

In [ ]:
pctl_df

In [ ]:
# Create subplots for each label in 2x3 grid
n_labels = len(pctl_df)

# Calculate grid dimensions
nrows = 2
ncols = 4
total_subplots = nrows * ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(12, 8))

# Flatten axes array for easier indexing
axes = axes.flatten()

# Create a subplot for each label
for idx, (label_idx, row) in enumerate(pctl_df.iterrows()):
    if idx >= total_subplots:
        break

    ax = axes[idx]

    # Extract percentile values for this label
    pctl_data = []
    pctl_names = []
    for pctl in pctl_values:
        pctl_data.append(row[f"pctl_{pctl}"])
        pctl_names.append(f"{pctl}%")

    # Create bar plot for this label with color
    bars = ax.bar(
        pctl_names,
        pctl_data,
        color=plt.cm.Set3(idx),
        alpha=0.7,
        edgecolor="black",
        linewidth=0.5,
    )

    # Calculate y-axis range for proper annotation positioning
    y_max = max(pctl_data)
    y_range = y_max - 0  # since we set bottom=0

    # Set y-axis limits with extra space for annotations
    ax.set_ylim(0, y_max * 1.15)

    # Annotate bars with values - always place text above the bars
    for i, (bar, value) in enumerate(zip(bars, pctl_data)):
        # Always position text above the bar
        text_y = bar.get_height() + y_range * 0.02
        text_color = "black"

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            text_y,
            f"{value:.0f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            color=text_color,
        )

    # Calculate subplot position in grid
    row_pos = idx // ncols
    col_pos = idx % ncols

    # Customize subplot
    ax.set_title(f"{row['label']}", fontsize=12, fontweight="bold")

    # Only show x-axis label on bottom row
    if row_pos == nrows - 1:
        ax.set_xlabel("Percentiles", fontsize=10)
    else:
        ax.set_xlabel("")

    # Only show y-axis label on leftmost column
    if col_pos == 0:
        ax.set_ylabel("Total Packets per Session", fontsize=10)
    else:
        ax.set_ylabel("")

    # Add grid for better readability
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.set_axisbelow(True)

    # Rotate x-axis labels if needed
    ax.tick_params(axis="x", rotation=0)

    # Set y-axis to start from 0
    ax.set_ylim(bottom=0)

# Hide unused subplots
for idx in range(n_labels, total_subplots):
    axes[idx].axis("off")


# Adjust layout with proper spacing
plt.tight_layout(rect=[0, 0.02, 1, 0.92])
plt.savefig(
    r"rosaid\assets\figures\total_packets_per_session_percentiles_by_label_iec104.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


### Find Image Height

In [ ]:
labelled_df = pd.read_csv(
    r"rosaid\data\iec104_sessions\labelled_sessions.csv"
)
combined_df = pd.read_csv(
    r"rosaid\data\iec104_combined.csv"
)
labelled_df.shape, combined_df.shape

In [ ]:
labelled_df.columns

In [ ]:
import numpy as np

(
    np.percentile(labelled_df.total_matched_pkts.values.tolist(), 99),
    np.percentile(labelled_df.total_labeled_pkts.values.tolist(), 99),
)

## ROSIDS23

### View Data and Make Pair

In [ ]:
data_root = Path(r"F:\work\dataset\rosid23")
pcap_files = list(data_root.glob("**/*.pcap"))
csv_files = list(data_root.glob("**/*.csv"))
mapping = {
    "DoS": "DoS",
    "noattack": "normal",
    "subcriberflood": "subscriberflood",
    "unauthorizedpublisher": "unauthorizedpublisher",
    "unauthorizedsubsriber": "unauthorizedsubscriber",
}

[p.stem for p in pcap_files], [c.stem for c in csv_files]

In [ ]:
# save mapping to json
import json

with open(
    r"rosaid\assets\rosid23_pcap_csv_mapping.json",
    "w",
) as f:
    json.dump(mapping, f, indent=4)

In [ ]:
lbl_cnts = {}
for sample_pcap in pcap_files:
    csv_file = data_root / (mapping[sample_pcap.stem] + ".csv")
    print(csv_file)
    df = pd.read_csv(csv_file)
    lbl_cnts[sample_pcap.stem] = df.Label.value_counts().to_dict()

    print(df.Label.value_counts())

In [ ]:
cnts = {}
for k, v in lbl_cnts.items():
    for lbl, cnt in v.items():
        if lbl not in cnts:
            cnts[lbl] = 0
        cnts[lbl] += cnt
cnts

It seems all the csv files related to the pcaps contain two labels: 1 being attack 0 being non attack. 

While generating images:
- if the label is 1, then it is attack else it is normal/benign

In [ ]:
combined_df = pd.read_csv(data_root / "ROSIDS23.csv")
combined_df.head()

In [ ]:
combined_df.Label.value_counts()

In [ ]:
# find pctl of total packets per session
combined_df["total_pkts"] = combined_df["Tot Fwd Pkts"] + combined_df["Tot Bwd Pkts"]
combined_df["total_pkts"].describe()

### Total Packets 

In [ ]:
import numpy as np

pctl_values = [80, 90, 93, 95, 99]
# for all labels find pctl values
pctl_df = pd.DataFrame(
    columns=[
        "label",
        "num_sessions",
        *[f"pctl_{pctl}" for pctl in pctl_values],
        "max_pkts",
        *[f"nsess_pctl_{pctl}" for pctl in pctl_values],
    ]
)

for key, group in combined_df.groupby("Label"):
    pctl_row = [key, len(group)]
    nsess = []
    for pctl in pctl_values:
        value = np.percentile(group["total_pkts"], pctl)
        pctl_row.append(value)
        nsess.append(len(group[group["total_pkts"] <= value]))

    pctl_row.append(group["total_pkts"].max())
    pctl_row.extend(nsess)
    pctl_df.loc[len(pctl_df)] = pctl_row

# add for all labels combined
pctl_row = ["All Labels", len(combined_df)]
nsess = []
for pctl in pctl_values:
    value = np.percentile(combined_df["total_pkts"], pctl)
    pctl_row.append(value)
    nsess.append(len(combined_df[combined_df["total_pkts"] <= value]))
pctl_row.append(combined_df["total_pkts"].max())
pctl_row.extend(nsess)
pctl_df.loc[len(pctl_df)] = pctl_row

In [ ]:
pctl_df

In [ ]:
# Create subplots for each label in 2x3 grid
n_labels = len(pctl_df)

# Calculate grid dimensions
nrows = 2
ncols = 3
total_subplots = nrows * ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(12, 8))

# Flatten axes array for easier indexing
axes = axes.flatten()

# Create a subplot for each label
for idx, (label_idx, row) in enumerate(pctl_df.iterrows()):
    if idx >= total_subplots:
        break

    ax = axes[idx]

    # Extract percentile values for this label
    pctl_data = []
    pctl_names = []
    for pctl in pctl_values:
        pctl_data.append(row[f"pctl_{pctl}"])
        pctl_names.append(f"{pctl}%")

    # Create bar plot for this label with color
    bars = ax.bar(
        pctl_names,
        pctl_data,
        color=plt.cm.Set3(idx),
        alpha=0.7,
        edgecolor="black",
        linewidth=0.5,
    )

    # Calculate y-axis range for proper annotation positioning
    y_max = max(pctl_data)
    y_range = y_max - 0  # since we set bottom=0

    # Set y-axis limits with extra space for annotations
    ax.set_ylim(0, y_max * 1.15)

    # Annotate bars with values - always place text above the bars
    for i, (bar, value) in enumerate(zip(bars, pctl_data)):
        # Always position text above the bar
        text_y = bar.get_height() + y_range * 0.02
        text_color = "black"

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            text_y,
            f"{value:.0f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            color=text_color,
        )

    # Calculate subplot position in grid
    row_pos = idx // ncols
    col_pos = idx % ncols

    # Customize subplot
    ax.set_title(f"{row['label']}", fontsize=12, fontweight="bold")

    # Only show x-axis label on bottom row
    if row_pos == nrows - 1:
        ax.set_xlabel("Percentiles", fontsize=10)
    else:
        ax.set_xlabel("")

    # Only show y-axis label on leftmost column
    if col_pos == 0:
        ax.set_ylabel("Total Packets per Session", fontsize=10)
    else:
        ax.set_ylabel("")

    # Add grid for better readability
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.set_axisbelow(True)

    # Rotate x-axis labels if needed
    ax.tick_params(axis="x", rotation=0)

    # Set y-axis to start from 0
    ax.set_ylim(bottom=0)

# Hide unused subplots
for idx in range(n_labels, total_subplots):
    axes[idx].axis("off")


# Adjust layout with proper spacing
plt.tight_layout(rect=[0, 0.02, 1, 0.92])
plt.savefig(
    r"rosaid\assets\figures\total_packets_per_session_percentiles_by_label_rosids23.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


In [ ]:
# Create subplots for each label in 2x3 grid
n_labels = len(pctl_df)

# Calculate grid dimensions
nrows = 2
ncols = 3
total_subplots = nrows * ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(12, 8))

# Flatten axes array for easier indexing
axes = axes.flatten()

# Create a subplot for each label
for idx, (label_idx, row) in enumerate(pctl_df.iterrows()):
    if idx >= total_subplots:
        break

    ax = axes[idx]

    # Extract percentile values for this label
    pctl_data = []
    pctl_names = []
    for pctl in pctl_values:
        pctl_data.append(row[f"nsess_pctl_{pctl}"])
        pctl_names.append(f"{pctl}%")

    # Create bar plot for this label with color
    bars = ax.bar(
        pctl_names,
        pctl_data,
        color=plt.cm.Set3(idx),
        alpha=0.7,
        edgecolor="black",
        linewidth=0.5,
    )

    # Calculate y-axis range for proper annotation positioning
    y_max = max(pctl_data)
    y_range = y_max - 0  # since we set bottom=0

    # Set y-axis limits with extra space for annotations
    ax.set_ylim(0, y_max * 1.15)

    # Annotate bars with values - always place text above the bars
    for i, (bar, value) in enumerate(zip(bars, pctl_data)):
        # Always position text above the bar
        text_y = bar.get_height() + y_range * 0.02
        text_color = "black"

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            text_y,
            f"{value:.0f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            color=text_color,
        )

    # Calculate subplot position in grid
    row_pos = idx // ncols
    col_pos = idx % ncols

    # Customize subplot
    ax.set_title(f"{row['label']}", fontsize=12, fontweight="bold")

    # Only show x-axis label on bottom row
    if row_pos == nrows - 1:
        ax.set_xlabel("Percentiles", fontsize=10)
    else:
        ax.set_xlabel("")

    # Only show y-axis label on leftmost column
    if col_pos == 0:
        ax.set_ylabel("Total Packets per Session", fontsize=10)
    else:
        ax.set_ylabel("")

    # Add grid for better readability
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.set_axisbelow(True)

    # Rotate x-axis labels if needed
    ax.tick_params(axis="x", rotation=0)

    # Set y-axis to start from 0
    ax.set_ylim(bottom=0)

# Hide unused subplots
for idx in range(n_labels, total_subplots):
    axes[idx].axis("off")


# Adjust layout with proper spacing
plt.tight_layout(rect=[0, 0.02, 1, 0.92])
plt.savefig(
    r"rosaid\assets\figures\total_session_after_percentiles_by_label_rosids23.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


In [ ]:
109767 / 136681

In [ ]:
df = pd.read_csv(r"F:\work\dataset\rosid23\unauthorizedpublisher.csv")

df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df.sort_values(by="Timestamp", inplace=True)
df["num_packets"] = df["Tot Fwd Pkts"] + df["Tot Bwd Pkts"]
df = df.query("num_packets <= 4378")


In [ ]:
df["Timestamp"].max() - df["Timestamp"].min()

In [ ]:
df[
    [
        "Flow ID",
        "Src IP",
        "Src Port",
        "Dst IP",
        "Dst Port",
        "Protocol",
        "Timestamp",
        "Flow Duration",
        "num_packets",
        "Label",
    ]
].head(20)


### Find timing

First see the first packet's time in PCAP and the first row's time in CSV file.

In [ ]:
import numpy as np


pctl_values = []

for key, value in mapping.items():
    print(key, value)
    pcap_file = key + ".pcap"
    pcap_path = data_root / pcap_file
    csv_file = data_root / (value + ".csv")
    df = pd.read_csv(csv_file)
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    df.sort_values(by="Timestamp", inplace=True)
    df["num_packets"] = df["Tot Fwd Pkts"] + df["Tot Bwd Pkts"]

    # plot boxplot of num_packets
    # plt.figure(figsize=(10, 6))
    # sns.boxplot(y=df["num_packets"])
    # plt.title(f"Number of Packets per Session - {value.upper()}")
    # plt.ylabel("Number of Packets")
    # plt.show()
    # print 99 percentile of num_packets
    curr_pctl = []
    for pctl in [90, 95, 99]:
        value = np.percentile(df["num_packets"], pctl)
        curr_pctl.append(value)
    curr_pctl.extend([value, len(df)])
    pctl_values.append(curr_pctl)
    print(
        f"{pctl}th percentile of number of packets: {value} and max: {df['num_packets'].max()} | time diff: {df.Timestamp.max() - df.Timestamp.min()}"
    )

    curr_packets = next(stream_pcap(str(pcap_path)))
    pcap_time = pd.to_datetime(float(curr_packets[1]) * 10**9)
    time_diff = df.Timestamp.min() - pcap_time
    print(
        f"DF first packet time: {df.Timestamp.min()}, PCAP first packet time: {pcap_time}, Difference: {time_diff}"
    )

In [ ]:
pctl_df = pd.DataFrame(
    pctl_values,
    columns=[
        "90th_percentile",
        "95th_percentile",
        "99th_percentile",
        "max_packets",
        "total_sessions",
    ],
    index=list(mapping.keys()),
)

In [ ]:
pctl_df

It seems the timing in PCAP is 3 hours late. It could be due to timezone. So for easier, we could subtract 3 hours from df.

In [ ]:
df[
    [
        "Flow ID",
        "Src IP",
        "Src Port",
        "Dst IP",
        "Dst Port",
        "Protocol",
        "Timestamp",
        "Flow Duration",
        "num_packets",
        "Label",
    ]
].head(20)

In [ ]:
df.columns

### Read CSV

In [ ]:
import pandas as pd

df = pd.read_csv(r"F:/work/dataset/rosid23/DoS.csv")
df["total_pkts"] = df["Tot Fwd Pkts"] + df["Tot Bwd Pkts"]

df

### Visualisation Feature importance

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

pfi_data_root = Path(
    r"rosaid\results\pfi_results"
)
for data_name in pfi_data_root.iterdir():
    f1_scores = pd.read_csv(data_name / "f1_scores.csv")
    top_features = pd.read_csv(data_name / "top_features.csv")
    f1_scores["feature"] = top_features["feature"]
    f1_scores["importance"] = top_features["importance"]

    # write all feature names in txt file
    with open(data_name / "selected_features.txt", "w") as f:
        for feature in f1_scores["feature"].tolist():
            f.write(f"'{feature}',")

    # find the first point where the f1 score is not increasing significantly
    prev_f1 = 0.0
    optimal_num_features = 1
    threshold = 0.00001  # define a threshold for significant increase
    start_f1 = 0.999
    for index, row in f1_scores.iterrows():
        curr_f1 = row["f1_score"]
        if curr_f1 - prev_f1 < threshold and curr_f1 >= start_f1:
            optimal_num_features = row["num_features"]
            break
        prev_f1 = curr_f1
    logger.info(
        f"Optimal number of features for {data_name.name}: {optimal_num_features}"
    )
    # print features till optimal num features
    logger.info(
        f"Selected features: {f1_scores[f1_scores['num_features'] <= optimal_num_features]['feature'].tolist()}"
    )
    # plot number of features vs f1 score vs importance (in second y axis)
    plt.figure(figsize=(14, 6))
    ax1 = plt.gca()
    ax2 = ax1.twinx()
    ax1.plot(
        f1_scores["num_features"],
        f1_scores["f1_score"],
        color="blue",
        marker="o",
        label="F1 Score",
    )
    ax2.plot(
        f1_scores["num_features"],
        f1_scores["importance"],
        color="red",
        marker="x",
        label="Feature\nImportance",
    )
    ax1.set_xlabel("Features")
    ax1.set_ylabel("F1 Score", color="blue")
    ax2.set_ylabel("Feature Importance", color="red")
    plt.title(
        f"Feature Importance vs F1 Score - {data_name.name.upper().split('_')[0]}"
    )
    # in x, show feature names but if too many characters, newline
    feature_names = f1_scores["feature"].tolist()
    fnames = []
    # split_from = 5
    # for fname in feature_names:
    #     if len(fname) > split_from:
    #         fname = "\n".join(
    #             [fname[i : i + split_from] for i in range(0, len(fname), split_from)]
    #         )
    #     fnames.append(fname)
    # ax1.set_xticks(f1_scores["num_features"])
    # ax1.set_xticklabels(fnames, rotation=0, ha="right")

    # plot the optimal f1 increase point
    optimal_row = f1_scores.iloc[optimal_num_features - 1]
    ax1.axvline(
        x=optimal_row["num_features"],
        color="green",
        linestyle="--",
        label="Optimal Num Features",
    )

    # Calculate the center of the y-axis for text positioning
    y_min, y_max = ax1.get_ylim()
    y_center = (y_min + y_max) / 2

    # write f1, num features, importance at center of the line
    ax1.text(
        optimal_row["num_features"] + 0.5,
        y_center,  # Use calculated center instead of fixed 0.5
        f"Num Features: {optimal_row['num_features']}\nF1 Score: {optimal_row['f1_score']:.4f}",
        color="green",
        fontsize=9,
        ha="left",
        va="center",  # Vertically center the text
        bbox=dict(
            facecolor="white", alpha=0.8, edgecolor="green", boxstyle="round,pad=0.3"
        ),
    )
    ax1.legend(loc="upper left")
    ax2.legend(loc="upper right")
    # show grid
    ax1.grid(True)
    plt.tight_layout()
    plt.savefig(
        data_name / "feature_importance_vs_f1_score.pdf",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()  # Add this to display the plot


### Image Height

In [ ]:
lbl_df = pd.read_csv(
    r"rosaid\data\rosids23_sessions\labelled_sessions.csv"
)
lbl_df["start_time"] = pd.to_datetime(lbl_df["start_time"]) + pd.Timedelta(hours=3)
lbl_df["start_ts"] = lbl_df.start_time.astype("int64") // 10**9

orig_df = pd.read_csv(r"F:/work/dataset/rosid23/ROSIDS23.csv")
orig_df["Label"] = orig_df["Label"].str.upper()
orig_df["ts"] = pd.to_datetime(orig_df["Timestamp"])
orig_df["epoch_time"] = orig_df.ts.astype("int64") // 10**9

lbl_df.shape, orig_df.shape, len(orig_df) - len(lbl_df)

In [ ]:
lbl_df.start_ts

In [ ]:
orig_df.epoch_time, lbl_df.start_timestamp

In [ ]:
pd.merge(
    lbl_df,
    orig_df,
    left_on=[
        # "flow_id",
        "start_timestamp",
        "flow_label",
    ],
    right_on=[
        # "Flow ID",
        "epoch_time",
        "Label",
    ],
    how="inner",
    indicator=True,
)

In [ ]:
lbl_df.columns

In [ ]:
# check the describe of total_pkts
lbl_df.total_labeled_pkts.describe(), lbl_df.total_matched_pkts.describe()

In [ ]:
lbl_df.groupby("flow_label")["raw_bytes_max_length"].describe()

In [ ]:
import numpy as np

(
    np.percentile(lbl_df.total_labeled_pkts.values.tolist(), 99),
    np.percentile(lbl_df.total_matched_pkts.values.tolist(), 99),
)

In [ ]:
np.percentile(
    lbl_df.query('flow_label=="BENIGN"').total_labeled_pkts.values.tolist(), 99
)

### Total Labeled PKTS Box

In [ ]:
# plt.figure(figsize=(3, 2))

# Create a boxplot of total_labeled_pkts grouped by flow_label
lbl_df.boxplot(column="total_labeled_pkts", by="flow_label", figsize=(6, 4))

# plt.title("Total Labeled Packets per Flow Label")
plt.suptitle("")  # Remove automatic pandas suptitle
plt.xlabel("Flow Label")
plt.ylabel("Total Labeled Packets")
# plt.xticks(rotation=45)
plt.title("")
plt.tight_layout()
plt.savefig(
    r"rosaid\assets\figures\total_labeled_packets_per_flow_label_rosids23.pdf",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


### Plot Class Distribution

In [ ]:
# Data
labels = ["Benign", "DoS", "SubFlood", "UnauthPub", "UnauthSub"]
counts = [62511, 31000, 30064, 7817, 5289]

plt.figure(figsize=(8, 5))
bars = plt.bar(
    labels,
    counts,
    color=["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"],
    edgecolor="black",
)

# Annotate bar values
for bar, value in zip(bars, counts):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{value}",
        ha="center",
        va="bottom",
        fontsize=16,
    )

plt.ylabel("Counts", fontdict={"size": 16})
plt.xlabel("Class", fontdict={"size": 16})
# tick font size
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
# plt.title("Class Distribution")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(
    r"rosaid\assets\figures\num_sessions_per_flow_label_rosids23.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


### Plot Images

In [ ]:
import cv2
from pathlib import Path

image_root = Path(
    r"rosaid\results\evaluation\adversarial_evaluation"
)
labels = ["BENIGN", "DOS", "SUBFLOOD", "UNAUTHPUB", "UNAUTHSUB"]

lbl_images = {}
for label in labels:
    for img_path in image_root.rglob(f"{label}_clean.png"):
        img = cv2.imread(str(img_path))
        lbl_images[label] = img
        break


In [ ]:
# plot 1 by 5 grid of images with label titles
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, label in zip(axes, labels):
    ax.imshow(cv2.cvtColor(lbl_images[label], cv2.COLOR_BGR2RGB))
    ax.set_title(label, fontsize=16)
    ax.axis("off")
plt.tight_layout()
plt.savefig(
    r"rosaid\assets\figures\ROSIDS23_images.pdf",
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
normal = cv2.imread(
    r"rosaid\assets\figures\subscriberflood.png"
)[:100]
normalized_img = cv2.imread(
    r"rosaid\assets\figures\subscriberflood_normalized.png"
)[:100]

plt.figure(figsize=(5, 5))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(normal, cv2.COLOR_BGR2RGB))
plt.title("Session Packets Image (SPI)", fontsize=14)
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(normalized_img, cv2.COLOR_BGR2RGB))
plt.title("Session Packets\nPayload Image (SPPI)", fontsize=14)
plt.axis("off")
plt.tight_layout()
plt.savefig(
    r"rosaid\assets\figures\subcriberflood_vs_normalized.pdf",
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
from rosaid.utils.vis import subplot_images
import numpy as np

image = normal.copy()
filtered_image = image.copy()
non_zero_cols = image.sum(axis=0) != 0
filtered_image = image[:, non_zero_cols].copy()

new_image = []
freq_map = []

for row in range(filtered_image.shape[0]):
    row_data = filtered_image[row, :]
    if sum(row_data) == 0:
        continue

    # find leftmost and rightmost non zero pixels
    leftmost = np.argmax(row_data != 0)
    rightmost = len(row_data) - np.argmax(row_data[::-1] != 0)
    row_data_cropped = row_data[leftmost:rightmost]
    # now calculate frequency of each pixel value
    unique, counts = np.unique(row_data_cropped, return_counts=True)

    # normalze the frequency to 0-255
    # if unique[0] == 0:
    #     counts[0] = 0  # ignore black pixel frequency
    norm_counts = (counts / counts.max() * 255).astype(np.uint8)

    new_row = np.zeros(256)
    new_row[unique] = counts
    # fill new_image row with pixel values according to frequency
    norm_row = np.zeros(256, dtype=np.uint8)
    norm_row[unique] = norm_counts
    new_image.append(norm_row)
    freq_map.append(new_row)

new_image = np.array(new_image, dtype=np.uint8)
freq_map = np.array(freq_map)

subplot_images(
    [image, filtered_image, new_image],
    titles=["Original Image", "Filtered Image", "Frequency Image"],
    show=False,
)

In [ ]:
def freq_norm_image(image: np.ndarray) -> np.ndarray:
    filtered_image = image.copy()
    non_zero_cols = image.sum(axis=0) != 0
    filtered_image = image[:, non_zero_cols].copy()

    new_image = []
    freq_map = []

    for row in range(filtered_image.shape[0]):
        row_data = filtered_image[row, :]
        if sum(row_data) == 0:
            continue

        # find leftmost and rightmost non zero pixels
        leftmost = np.argmax(row_data != 0)
        rightmost = len(row_data) - np.argmax(row_data[::-1] != 0)
        row_data_cropped = row_data[leftmost:rightmost]
        # now calculate frequency of each pixel value
        unique, counts = np.unique(row_data_cropped, return_counts=True)

        # normalze the frequency to 0-255
        # if unique[0] == 0:
        #     counts[0] = 0  # ignore black pixel frequency
        norm_counts = (counts / counts.max() * 255).astype(np.uint8)

        new_row = np.zeros(256)
        new_row[unique] = counts
        # fill new_image row with pixel values according to frequency
        norm_row = np.zeros(256, dtype=np.uint8)
        norm_row[unique] = norm_counts
        new_image.append(norm_row)
        freq_map.append(new_row)
    freq_map = np.array(freq_map)
    freq_mean = freq_map.mean(axis=1)
    freq_std = freq_map.std(axis=1)
    freq_zscore = (freq_map - freq_mean[:, None]) / freq_std[:, None]
    freq_zscore_img = np.nan_to_num(freq_zscore)

    mean_img = freq_zscore_img.mean(axis=0).reshape(16, 16)
    mean_img = (mean_img - mean_img.min()) / (mean_img.max() - mean_img.min()) * 255
    std_img = freq_zscore_img.std(axis=0).reshape(16, 16)
    std_img = (std_img - std_img.min()) / (std_img.max() - std_img.min()) * 255
    median_img = np.median(freq_zscore_img, axis=0).reshape(16, 16)
    median_img = (
        (median_img - median_img.min()) / (median_img.max() - median_img.min()) * 255
    )

    # 16x16x3 images
    sess_img = np.stack([mean_img, std_img, median_img], axis=2).astype(np.uint8)

    return sess_img


sess_img = freq_norm_image(normal)
subplot_images(
    [image, filtered_image, sess_img],
    titles=["Original Image", "Filtered Image", "Z-Score Frequency Image"],
    show=False,
)

### Read PCAP

In [ ]:
# pcap_pth = r"F:\work\dataset\rosid23\noattack.pcap"
pcap_pth = r"rosaid\data\dnp3data\DNP3_Intrusion_Detection_Dataset_Final\20200515_DNP3_Warm_Restart_Attack\DNP3 PCAP Files\20200515_DNP3_Warm_Restart_Attack_UOWM_DNP3_Dataset_Attacker_02.pcap"

for pkt in stream_pcap(pcap_pth):
    print(pkt)
    break

In [ ]:
def anonymize_packet(packet):
    """Anonymize packet by removing address information"""
    # Create copy to avoid modifying original packet
    pkt = packet.copy()

    # IP layer handling
    if pkt.haslayer(IP):
        pkt[IP].src = "0.0.0.0"
        pkt[IP].dst = "0.0.0.0"

    # Ethernet layer handling
    if pkt.haslayer(Ether):
        pkt[Ether].src = "00:00:00:00:00:00"
        pkt[Ether].dst = "00:00:00:00:00:00"

    # TCP layer handling
    if pkt.haslayer(TCP):
        pkt[TCP].sport = 0
        pkt[TCP].dport = 0

    return pkt


In [ ]:
pkt[0].show()

In [ ]:
raw(pkt[0].load)

In [ ]:
np.frombuffer(raw(pkt[0].load), dtype=np.uint8)


In [ ]:
byte_counts = np.zeros(256, dtype=np.float32)
for byte_val in bytes(pkt[0].payload):
    byte_counts[byte_val] += 1

In [ ]:
bytes(pkt[0])

In [ ]:
def freq_normalize_payload(packet) -> np.ndarray:
    """Frequency normalize packet payload"""
    payload = raw(packet.load)
    byte_array = np.frombuffer(payload, dtype=np.uint8)

    # Calculate frequency of each byte value (0-255)
    unique, counts = np.unique(byte_array, return_counts=True)

    # Normalize frequencies to 0-255
    norm_counts = (counts / counts.max() * 255).astype(np.uint8)

    # Create frequency normalized array
    freq_norm_array = np.zeros(256, dtype=np.uint8)
    freq_norm_array[unique] = norm_counts

    return freq_norm_array


freq_norm_payload = freq_normalize_payload(pkt[0])
freq_norm_payload

In [ ]:
anpkt = raw(anonymize_packet(pkt[0]))
anpkt

In [ ]:
np.frombuffer(anpkt, dtype=np.uint8)

## 5G
https://zenodo.org/records/14811122

In [ ]:
df = pd.read_csv(
    r"rosaid\data\Network_Flows_Annotated.csv"
)

df["timestamp"] = pd.to_datetime(df["Timestamp"])

In [ ]:
from scapy.all import Ether, raw, rdpcap

pkts = rdpcap(r"rosaid\data\5g_first.pcap")

In [ ]:
pd.to_datetime(float(pkts[0].time) * 10**9)

In [ ]:
df["timestamp"].iloc[0] - pd.Timedelta(hours=2)

## Session Images

### Filtered

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from rosaid.utils.vis import subplot_images


def get_filtered_image(image: np.ndarray):
    """
    First, delete all the columns that is complete black.
    """
    filtered_image = image.copy()
    non_zero_cols = image.sum(axis=0) != 0
    filtered_image = image[:, non_zero_cols].copy()

    return filtered_image


### Normalized Frequency

In [ ]:
def image_to_normalized_frequency_image(
    filtered_image: np.ndarray,
):
    new_image = []

    for row in range(filtered_image.shape[0]):
        row_data = filtered_image[row, :]
        if sum(row_data) == 0:
            continue

        # find leftmost and rightmost non zero pixels
        leftmost = np.argmax(row_data != 0)
        rightmost = len(row_data) - np.argmax(row_data[::-1] != 0)
        row_data_cropped = row_data[leftmost:rightmost]
        # now calculate frequency of each pixel value
        unique, counts = np.unique(row_data_cropped, return_counts=True)
        # normalze the frequency to 0-255
        # if unique[0] == 0:
        #     counts[0] = 0  # ignore black pixel frequency
        counts = (counts / counts.max() * 255).astype(np.uint8)

        # fill new_image row with pixel values according to frequency
        new_row = np.zeros(256, dtype=np.uint8)
        new_row[unique] = counts
        new_image.append(new_row)
    new_image = np.array(new_image, dtype=np.uint8)

    return new_image

### Z-Image

In [ ]:
def sess_zscore_image(image: np.ndarray) -> np.ndarray:
    filtered_image = image[:, image.sum(axis=0) != 0].copy()
    new_image = []

    for row in range(filtered_image.shape[0]):
        row_data = filtered_image[row]
        if row_data.sum() == 0:
            continue

        leftmost = np.argmax(row_data != 0)
        rightmost = len(row_data) - np.argmax(row_data[::-1] != 0)
        row_data_cropped = row_data[leftmost:rightmost]
        unique, counts = np.unique(row_data_cropped, return_counts=True)

        norm_counts = counts / counts.max() * 255
        norm_row = np.zeros(256, dtype=np.uint8)
        norm_row[unique] = norm_counts
        new_image.append(norm_row)

    freq_map = np.array(new_image)
    freq_mean = freq_map.mean(axis=1)
    freq_std = freq_map.std(axis=1)
    freq_zscore = np.nan_to_num(
        (freq_map - freq_mean[:, None]) / (freq_std[:, None] + 1e-8)
    )

    def safe_norm(img):
        img = img.astype(np.float32)
        img_min, img_max = img.min(), img.max()
        if img_max == img_min or not np.isfinite(img_max - img_min):
            return np.zeros_like(img, dtype=np.uint8)
        return (img - img_min) / (img_max - img_min) * 255

    mean_img = safe_norm(freq_zscore.mean(axis=0).reshape(16, 16))
    std_img = safe_norm(freq_zscore.std(axis=0).reshape(16, 16))
    median_img = safe_norm(np.median(freq_zscore, axis=0).reshape(16, 16))

    return np.stack([mean_img, std_img, median_img], axis=2)


def sess_zscore_gram_image(image: np.ndarray) -> np.ndarray:
    filtered_image = image[:, image.sum(axis=0) != 0].copy()
    new_image = []

    for row in range(filtered_image.shape[0]):
        row_data = filtered_image[row]
        if row_data.sum() == 0:
            continue

        leftmost = np.argmax(row_data != 0)
        rightmost = len(row_data) - np.argmax(row_data[::-1] != 0)
        row_data_cropped = row_data[leftmost:rightmost]
        unique, counts = np.unique(row_data_cropped, return_counts=True)

        norm_counts = (counts / counts.max() * 255).astype(np.uint8)
        norm_row = np.zeros(256, dtype=np.uint8)
        norm_row[unique] = norm_counts
        new_image.append(norm_row)

    freq_map = np.array(new_image)
    freq_mean = freq_map.mean(axis=1)
    freq_std = freq_map.std(axis=1)
    freq_zscore = np.nan_to_num(
        (freq_map - freq_mean[:, None]) / (freq_std[:, None] + 1e-8)
    )

    def safe_norm(img):
        img = img.astype(np.float32)
        img_min, img_max = img.min(), img.max()
        if img_max == img_min or not np.isfinite(img_max - img_min):
            return np.zeros_like(img, dtype=np.uint8)
        return (img - img_min) / (img_max - img_min) * 255

    mean_img = safe_norm(freq_zscore.mean(axis=0)).reshape(-1, 1)
    std_img = safe_norm(freq_zscore.std(axis=0)).reshape(-1, 1)
    median_img = safe_norm(np.median(freq_zscore, axis=0)).reshape(-1, 1)

    # gram of each img: 256, 256
    stkd = np.stack([mean_img, std_img, median_img], axis=2)
    gram = stkd * stkd.transpose(1, 0, 2)  # 256, 256, 3
    mat_gram = np.dot(stkd.reshape(256, 3), stkd.reshape(256, 3).T)  # 256, 256

    # min max both and to uint8
    gram_min, gram_max = gram.min(), gram.max()
    gram = (gram - gram_min) / (gram_max - gram_min) * 255
    mat_gram_min, mat_gram_max = mat_gram.min(), mat_gram.max()
    mat_gram = (mat_gram - mat_gram_min) / (mat_gram_max - mat_gram_min) * 255
    return gram, mat_gram


### Gram Image

In [ ]:
def gram_image(image: np.ndarray) -> np.ndarray:
    """Compute Gram matrix of the image"""
    pixels = image.astype(float)

    gram_matrix = np.dot(pixels.T, pixels)
    gram_matrix = (
        (gram_matrix - gram_matrix.min())
        / (gram_matrix.max() - gram_matrix.min())
        * 255
    )
    return gram_matrix


def gram_image_3d(image: np.ndarray, channel_norm: bool = True) -> np.ndarray:
    """Compute Gram matrix of the image"""
    pixels = image.astype(float)

    orig_gram = np.dot(pixels.T, pixels)
    row_rev_gram = np.dot(pixels.T, pixels)[::-1, :]
    col_rev_gram = np.dot(pixels.T, pixels)[:, ::-1]
    gram_3d = np.stack([orig_gram, row_rev_gram, col_rev_gram], axis=2)
    gram_matrix = gram_3d

    gram_matrix = (
        (gram_matrix - gram_matrix.min()) / (gram_matrix.max() - gram_matrix.min())
    ) * 255

    return gram_matrix


In [ ]:
image = cv2.imread(
    r"rosaid\assets\figures\subcriberflood.png"
)
image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# find first non zero column
first_nonzero_col = np.argmax(image.sum(axis=0) != 0)


filtered = get_filtered_image(image)


In [ ]:
filtered.max(), (np.dot(filtered.T.astype(float), filtered.astype(float)).max())

In [ ]:
res = []
filtered_gram, gram3dcn, gram3dncn = (
    gram_image(image[:, first_nonzero_col:]),
    gram_image_3d(image[:, first_nonzero_col:]),
    gram_image_3d(image[:, first_nonzero_col:], channel_norm=False),
)
print(filtered_gram.shape, gram3dcn.shape, image.shape)
subplot_images(
    [image, filtered, filtered_gram, gram3dcn, gram3dncn],
    titles=[
        "Original Image",
        "Filtered Image",
        "Gram Image",
        "Gram 3D Image",
        "Gram 3D Image (No Channel Norm)",
    ],
    ret_fig=False,
    show=False,
    order=(1, -1),
)

In [ ]:
image = cv2.imread(
    r"rosaid\assets\figures\subcriberflood.png"
)
image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)


def all_images(image: np.ndarray):
    plot_images = [image]
    titles = ["Original"]
    filtered_image = get_filtered_image(image)

    freq_image = image_to_normalized_frequency_image(filtered_image)

    z_image = sess_zscore_image(image)
    z_gram, z_gram1d = sess_zscore_gram_image(image)

    unfiltered_gram_img = gram_image(image)
    filtered_gram_img = gram_image(filtered_image)
    freq_gram_img = gram_image(freq_image)

    plot_images.extend(
        [
            filtered_image,
            freq_image,
            z_image,
            z_gram,
            z_gram1d,
            unfiltered_gram_img,
            filtered_gram_img,
            freq_gram_img,
            gram_image_3d(image),
            gram_image_3d(filtered_image),
        ]
    )
    titles.extend(
        [
            "Filt.",
            "NormFreq",
            "Z-Freq",
            "ZGram3D",
            "ZGram2D",
            "UnfiltGram",
            "FiltGram",
            "FreqGram",
            "UFGram3D",
            "FGram3D",
        ]
    )
    return plot_images, titles


plot_images, titles = all_images(image)
subplot_images(plot_images, titles=titles, order=(1, -1), ret_fig=False)


In [ ]:
import pandas as pd

root_dir = Path(
    r"rosaid\data\120_timeout_dnp3_sessions\session_images"
)
csv = pd.read_csv(
    r"rosaid\data\120_timeout_dnp3_sessions\labelled_sessions.csv"
)

# root_dir = Path(
#     r"rosaid\data\iec104_sessions\session_images"
# )
# csv = pd.read_csv(
#     r"rosaid\data\iec104_sessions\labelled_sessions.csv"
# )
# root_dir = Path(
#     r"rosaid\data\rosids23_sessions\session_images"
# )
# csv = pd.read_csv(
#     r"rosaid\data\rosids23_sessions\labelled_sessions.csv"
# )

data_type = root_dir.parent.name
csv.raw_bytes_max_length.describe(percentiles=[0.25, 0.5, 0.75, 0.85, 0.9, 0.95, 0.99])

In [ ]:
images = []
titles = []
for flow_label, group in csv.groupby("flow_label"):
    for idx, row in group.iterrows():
        img_path = root_dir / row.session_file_name.replace(".pcap", ".png")
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        if img.shape[0] == 1:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        flow_label = flow_label.replace("_", "\n")
        imgs, ttls = all_images(img)
        ttls[0] = f"{flow_label}"
        images.extend(imgs)
        titles.extend(ttls)

        break


In [ ]:
dtype = (
    "DNP3"
    if "dnp3" in data_type.lower()
    else "IEC104"
    if "iec104" in data_type.lower()
    else "ROSIDS23"
)
dtype = "image_" + dtype.lower()
save_path = f"{dtype}.pdf"
fig = subplot_images(
    images,
    titles=titles,
    order=(-1, 11),
    ret_fig=True,
)
fig.savefig(save_path, dpi=300, bbox_inches="tight")
